# Poisson Regression for Count Data

**Topics:** Poisson GLM, Count Data, Rate Models, Offset

## Overview

Model count outcomes (0, 1, 2, ...) using Poisson regression.

## What You'll Learn

- Fit Poisson GLM for count data
- Interpret coefficients as log-rate ratios
- Use offset for exposure/time
- Check for overdispersion
- Compare to linear regression

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.validation.metrics import root_mean_squared_error, mean_absolute_error

sns.set_style('whitegrid')
np.random.seed(42)

## Generate Count Data

Website visits per day as function of ad spend and content quality:

In [ ]:
n = 300

ad_spend = np.random.uniform(0, 10, n)  # $1000s
content_quality = np.random.uniform(1, 10, n)  # score

# Log rate
log_rate = (
    2.5  # baseline log visits
    + 0.15 * ad_spend
    + 0.08 * content_quality
)

# Expected count
lambda_true = np.exp(log_rate)

# Sample counts from Poisson
visits = np.random.poisson(lambda_true)

df = pd.DataFrame({
    'ad_spend': ad_spend,
    'content_quality': content_quality,
    'visits': visits,
    'lambda_true': lambda_true
})

print(f"Generated {n} days of website data")
print(f"\nVisits summary:")
print(df['visits'].describe())
print(f"\nMean: {df['visits'].mean():.2f}")
print(f"Variance: {df['visits'].var():.2f}")
print(f"Variance/Mean ratio: {df['visits'].var()/df['visits'].mean():.2f}")
print("  (Close to 1.0 suggests Poisson is appropriate)")

## Visualize Count Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution
axes[0].hist(df['visits'], bins=30, edgecolor='k', alpha=0.7)
axes[0].set_xlabel('Daily Visits')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Visit Counts')
axes[0].grid(alpha=0.3, axis='y')

# Ad spend vs visits
axes[1].scatter(df['ad_spend'], df['visits'], alpha=0.6, edgecolor='k', linewidth=0.5)
axes[1].set_xlabel('Ad Spend ($1000s)')
axes[1].set_ylabel('Daily Visits')
axes[1].set_title('Visits vs Ad Spend')
axes[1].grid(alpha=0.3)

# Content quality vs visits
axes[2].scatter(df['content_quality'], df['visits'], alpha=0.6, edgecolor='k', linewidth=0.5)
axes[2].set_xlabel('Content Quality Score')
axes[2].set_ylabel('Daily Visits')
axes[2].set_title('Visits vs Content Quality')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Fit Poisson Regression

In [ ]:
X = np.column_stack([
    np.ones(n),
    df['ad_spend'],
    df['content_quality']
])
y = df['visits'].values

result_poisson = fit_glm(X=X, y=y, family='poisson')

print("Poisson Regression Results:")
print(f"\nConverged: {result_poisson.converged_}")
print(f"Iterations: {result_poisson.n_iter_}")
print("\n" + "="*60)
print(result_poisson.summary())
print("="*60)

## Interpret Coefficients

In [ ]:
coef_names = ['Intercept', 'Ad Spend', 'Content Quality']
true_coefs = [2.5, 0.15, 0.08]

# Coefficients already include intercept when X has a column of ones
coefficients_full = result_poisson.coef_
p_values_full = result_poisson.p_values_

# Rate ratios (exp of coefficients)
rate_ratios = np.exp(coefficients_full)
true_rate_ratios = np.exp(true_coefs)

interp_df = pd.DataFrame({
    'Predictor': coef_names,
    'True Log-Rate': true_coefs,
    'Estimated Log-Rate': coefficients_full,
    'Rate Ratio': rate_ratios,
    'p-value': p_values_full
})

print("\nCoefficient Interpretation:")
print(interp_df.to_string(index=False))

print(f"\n📊 Interpretation:")
print(f"\n1. Ad Spend (β = {coefficients_full[1]:.4f}):")
print(f"   • Rate Ratio = {rate_ratios[1]:.4f}")
print(f"   • Each $1k increase → {(rate_ratios[1]-1)*100:.1f}% increase in visit rate")
print(f"\n2. Content Quality (β = {coefficients_full[2]:.4f}):")
print(f"   • Rate Ratio = {rate_ratios[2]:.4f}")
print(f"   • Each 1-point increase → {(rate_ratios[2]-1)*100:.1f}% increase in visit rate")

## Compare: Linear vs Poisson

In [ ]:
# Fit linear regression (Gaussian GLM)
result_linear = fit_glm(X=X, y=y, family='gaussian')

# Predictions
pred_poisson = result_poisson.predict(X)
pred_linear = result_linear.predict(X)

# Metrics
print("Model Comparison:")
print(f"\n{'Model':<15} {'RMSE':<10} {'MAE':<10}")
print("="*35)
print(f"{'Linear':<15} {root_mean_squared_error(y, pred_linear):<10.2f} {mean_absolute_error(y, pred_linear):<10.2f}")
print(f"{'Poisson':<15} {root_mean_squared_error(y, pred_poisson):<10.2f} {mean_absolute_error(y, pred_poisson):<10.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y, pred_linear, alpha=0.5, label='Linear', s=30)
axes[0].scatter(y, pred_poisson, alpha=0.5, label='Poisson', s=30)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Visits')
axes[0].set_ylabel('Predicted Visits')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals
axes[1].scatter(pred_linear, y - pred_linear, alpha=0.5, label='Linear', s=30)
axes[1].scatter(pred_poisson, y - pred_poisson, alpha=0.5, label='Poisson', s=30)
axes[1].axhline(0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Visits')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPoisson regression more appropriate for count data")
print("Linear regression can predict negative values (problematic!)")

# Check for negative predictions
neg_linear = (pred_linear < 0).sum()
neg_poisson = (pred_poisson < 0).sum()
print(f"\nNegative predictions: Linear={neg_linear}, Poisson={neg_poisson}")

## Check for Overdispersion

In [ ]:
# Dispersion parameter
residual_deviance = result_poisson.deviance_
df_resid = n - X.shape[1]
dispersion = residual_deviance / df_resid

print("Overdispersion Check:")
print(f"\nResidual deviance: {residual_deviance:.2f}")
print(f"Degrees of freedom: {df_resid}")
print(f"Dispersion parameter: {dispersion:.3f}")
print(f"\nInterpretation:")
if dispersion < 1.2:
    print("  ✓ No overdispersion (Poisson is appropriate)")
elif dispersion < 2.0:
    print("  ⚠ Mild overdispersion (consider quasi-Poisson or negative binomial)")
else:
    print("  ✗ Severe overdispersion (use negative binomial instead!)")

print(f"\n→ Empirical variance/mean: {df['visits'].var()/df['visits'].mean():.2f}")
print(f"→ Dispersion from model: {dispersion:.2f}")

## Offset Example

Model rate per unit time/exposure:

In [ ]:
# Simulate different observation periods
days_observed = np.random.choice([1, 2, 3, 5, 7], n)
df['days_observed'] = days_observed

# Total visits over observation period
df['total_visits'] = np.random.poisson(df['lambda_true'] * days_observed)

# Fit with offset
y_total = df['total_visits'].values
offset = np.log(days_observed)  # Log of exposure

result_offset = fit_glm(X=X, y=y_total, family='poisson', offset=offset)

# Coefficients already include intercept
coefficients_without_offset = result_poisson.coef_
coefficients_with_offset = result_offset.coef_

print("Poisson with Offset (modeling rate per day):")
print(f"\nCoefficients (should be similar to original model):")
comparison = pd.DataFrame({
    'Predictor': coef_names,
    'Without Offset': coefficients_without_offset,
    'With Offset': coefficients_with_offset
})
print(comparison.to_string(index=False))

print("\nOffset allows modeling rates when exposure varies")
print("Coefficients represent log rate per unit exposure (per day)")

## When to Use Poisson

- Count data (non-negative integers)
- Events in fixed time/space
- Variance ≈ Mean (equidispersion)
- No excess zeros

**If overdispersed:** See `03_count_data/02_negative_binomial.ipynb`  
**If excess zeros:** See `03_count_data/03_zero_inflated.ipynb`